In [1]:
#!/usr/bin/env python3
import sys 
sys.path.insert(0, '/uufs/chpc.utah.edu/common/home/koper-group4/bbaker/waveformArchive/gcc_build')
#sys.path.insert(0, '/uufs/chpc.utah.edu/common/home/koper-group1/bbaker/templateMatchingSource/rtseis/notchpeak4_gcc83_build/')
sys.path.insert(0, '/uufs/chpc.utah.edu/common/home/koper-group4/bbaker/mlmodels/intel_cpu_build')
sys.path.insert(0, '/uufs/chpc.utah.edu/common/home/koper-group4/bbaker/mlmodels/features/np4_build')
import h5py
import pyWaveformArchive as pwa 
import pyuussmlmodels as uuss
import pyuussFeatures as pf
# import libpyrtseis as rtseis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

import glob

# S-arrivals

In [2]:
archive_dir = '/uufs/chpc.utah.edu/common/home/koper-group4/bbaker/waveformArchive/archives/'
h5_archive_files = glob.glob(archive_dir + '/archive_????.h5')
catalog_dir = '/uufs/chpc.utah.edu/common/home/koper-group3/alysha/ben_catalogs/20240220'
arrival_catalog_3c = f'{catalog_dir}/currentEarthquakeArrivalInformation3CWithGains.csv'
startdate = datetime(2002, 1, 1, tzinfo=timezone.utc).timestamp()
print(f'Using events occuring on or after {startdate}')

print("Loading arrival catalog...")
arrival_catalog_df = pd.read_csv(arrival_catalog_3c, dtype = {'location' : object})
# Regress on Ml for S waves
arrival_catalog_df = arrival_catalog_df[ (arrival_catalog_df.phase == 'S') &
                                            (arrival_catalog_df.magnitude_type == 'l') &
                                            (arrival_catalog_df.origin_time >= startdate) ]
# # Focus on Yellowstone
arrival_catalog_df = arrival_catalog_df[ (arrival_catalog_df.event_lat > 44) &
                                            (arrival_catalog_df.event_lat < 45.167) &
                                            (arrival_catalog_df.event_lon > -111.333) &
                                            (arrival_catalog_df.event_lon < -109.75) ]

print("Opening archive files for reading...")
archive_manager = pwa.ArchiveManager()
archive_manager.open_files_for_reading(h5_archive_files)

Using events occuring on or after 1009843200.0
Loading arrival catalog...
Opening archive files for reading...


In [3]:
arrival_catalog_df["location"].value_counts()

location
01    18686
       5066
00      584
Name: count, dtype: int64

In [4]:
arrival_catalog_df.head()

,evid,network,station,location,channelz,channel1,channel2,phase,arrival_id,arrival_time,...,low_freq_corners_2,high_freq_corners_z,high_freq_corners_1,high_freq_corners_2,channel_dip_z,channel_azimuth_z,channel_dip_1,channel_azimuth_1,channel_dip_2,channel_azimuth_2
344,60000620,TA,H17A,01,BHZ,BHN,BHE,S,10013548,1.357546e+09,...,16.0,0.008330,0.008330,0.008330,-90.0,0.0,0.0,0.1,0.0,90.1
345,60000620,PB,B208,01,EHZ,EHN,EHE,S,10013551,1.357546e+09,...,40.0,0.003993,0.003993,0.003993,-90.0,0.0,0.0,30.0,0.0,120.0
349,60000622,WY,YHB,01,HHZ,HHN,HHE,S,10000492,1.349570e+09,...,40.0,4.204171,4.204171,3.564715,-90.0,0.0,0.0,0.0,0.0,90.0
352,60000622,WY,YHL,01,HHZ,HHN,HHE,S,10000494,1.349570e+09,...,40.0,4.204196,4.204196,4.204196,-90.0,0.0,0.0,0.0,0.0,90.0
353,60000622,WY,YMR,01,HHZ,HHN,HHE,S,10000497,1.349570e+09,...,40.0,4.204196,4.204196,4.204196,-90.0,0.0,0.0,0.0,0.0,90.0


In [5]:
arrival_catalog_df.columns

Index(['evid', 'network', 'station', 'location', 'channelz', 'channel1',
       'channel2', 'phase', 'arrival_id', 'arrival_time', 'pick_quality',
       'first_motion', 'take_off_angle', 'source_receiver_distance',
       'source_receiver_azimuth', 'travel_time_residual', 'receiver_lat',
       'receiver_lon', 'receiver_elev', 'event_lat', 'event_lon',
       'event_depth', 'origin_time', 'magnitude', 'magnitude_type', 'rflag',
       'gain_z', 'gain_1', 'gain_2', 'gain_units', 'low_freq_corners_z',
       'low_freq_corners_1', 'low_freq_corners_2', 'high_freq_corners_z',
       'high_freq_corners_1', 'high_freq_corners_2', 'channel_dip_z',
       'channel_azimuth_z', 'channel_dip_1', 'channel_azimuth_1',
       'channel_dip_2', 'channel_azimuth_2'],
      dtype='object')

# Split up the channe info

In [6]:
def add_approx_ondates(df, chan_or):
    gains_mindates = []
    gains_maxdates = []
    for i, row in df.iterrows():
        row = row[["network", "station", f"channel{chan_or}", "location", f"gain_{chan_or}", "gain_units"]]
        t_df = arrival_catalog_df[np.all(arrival_catalog_df[["network", "station", f"channel{chan_or}", "location", f"gain_{chan_or}", "gain_units"]] == row, axis=1)]
        mindate = datetime.fromtimestamp(t_df["arrival_time"].min(), tz=timezone.utc)
        maxdate = datetime.fromtimestamp(t_df["arrival_time"].max(), tz=timezone.utc)
        gains_mindates.append(mindate)
        gains_maxdates.append(maxdate)

    df["mindate"] = gains_mindates
    df["maxdate"] = gains_maxdates
    # df.head()

    return df

In [7]:
gainz_df = arrival_catalog_df[['network', 'station', 'location', 'channelz', 'gain_z', 'gain_units', 'channel_dip_z', 'channel_azimuth_z']].drop_duplicates()
gain1_df = arrival_catalog_df[['network', 'station', 'location', 'channel1', 'gain_1', 'gain_units', 'channel_dip_1', 'channel_azimuth_1']].drop_duplicates()
gain2_df = arrival_catalog_df[['network', 'station', 'location', 'channel2', 'gain_2', 'gain_units', 'channel_dip_2', 'channel_azimuth_2']].drop_duplicates()

gainz_df = add_approx_ondates(gainz_df, "z")
gain1_df = add_approx_ondates(gain1_df, "1")
gain2_df = add_approx_ondates(gain2_df, "2")

new_cols = ['network', 'station', 'location', 'channel', 'gain', 'gain_units', 'channel_dip', 'channel_azimuth', "mindate", "maxdate"]
gainz_df.columns = new_cols
gain1_df.columns = new_cols
gain2_df.columns = new_cols
gains_df = pd.concat([gainz_df, gain1_df, gain2_df]).sort_values(["network", "station", "location", "channel", "mindate"])
gains_df.head()

,network,station,location,channel,gain,gain_units,channel_dip,channel_azimuth,mindate,maxdate
12322,IW,FLWY,00,BH1,462429000.0,DU/M/S,0.0,0.0,2016-11-17 06:08:44.755000+00:00,2021-07-31 08:11:14.434572+00:00
47608,IW,FLWY,00,BH1,301720000.0,DU/M/S,0.0,2.8,2021-09-08 17:01:40.424286+00:00,2023-12-17 03:22:16.120020+00:00
12322,IW,FLWY,00,BH2,462429000.0,DU/M/S,0.0,90.0,2016-11-17 06:08:44.755000+00:00,2021-07-31 08:11:14.434572+00:00
47608,IW,FLWY,00,BH2,301720000.0,DU/M/S,0.0,92.8,2021-09-08 17:01:40.424286+00:00,2023-12-17 03:22:16.120020+00:00
12322,IW,FLWY,00,BHZ,462429000.0,DU/M/S,-90.0,0.0,2016-11-17 06:08:44.755000+00:00,2021-07-31 08:11:14.434572+00:00


In [8]:
gains_df[gains_df["station"] == "YMP"]

,network,station,location,channel,gain,gain_units,channel_dip,channel_azimuth,mindate,maxdate
18189,WY,YMP,01,HHE,5.352855e+08,DU/M/S,0.0,90.0,2015-12-07 11:06:32.258523+00:00,2023-03-29 14:10:41.556275+00:00
18189,WY,YMP,01,HHN,5.352855e+08,DU/M/S,0.0,0.0,2015-12-07 11:06:32.258523+00:00,2023-03-29 14:10:41.556275+00:00
18189,WY,YMP,01,HHZ,5.352855e+08,DU/M/S,-90.0,0.0,2015-12-07 11:06:32.258523+00:00,2023-03-29 14:10:41.556275+00:00


# Load in the database channel info

In [9]:
db_gains_df = pd.read_csv("/uufs/chpc.utah.edu/common/home/u1072028/PycharmProjects/seis-proc-db/data_files/db_channel_info.csv",  dtype = {'location' : object}).sort_values(["network", "station", "seed_code", "location"])
db_gains_df["location"] = db_gains_df["location"].fillna("  ")
db_gains_df.head()

,channel_id,network,station,location,seed_code,channel_samp_rate,sensit_units,sensit_freq,sensit_val,gain_vel,receiver_lat,receiver_lon,channel_azimuth,channel_ondate,channel_offdate
3,958,IW,FLWY,00,BH1,40.0,m/s,0.05,1.145990e+09,1.145993e+09,44.083002,-110.699888,0.0,2013-07-25 14:46:00,2016-10-22 04:00:00
4,568,IW,FLWY,00,BH1,40.0,m/s,0.02,4.624290e+08,4.624283e+08,44.083002,-110.699888,0.0,2016-10-22 04:00:00,2020-01-30 20:00:00
5,377,IW,FLWY,00,BH1,40.0,m/s,0.02,4.624290e+08,4.624283e+08,44.083002,-110.699888,0.0,2020-01-30 20:00:00,2021-08-29 18:00:00
6,4,IW,FLWY,00,BH1,40.0,m/s,0.02,2.939200e+08,2.939199e+08,44.083002,-110.699888,2.8,2021-08-29 18:00:00,NaN
7,959,IW,FLWY,00,BH2,40.0,m/s,0.05,1.145990e+09,1.145993e+09,44.083002,-110.699888,90.0,2013-07-25 14:46:00,2016-10-22 04:00:00


In [10]:
# db_gains_df["simple_gain_vel"] = db_gains_df.apply(lambda x: x["sensit_val"]*(2*np.pi*x["sensit_freq"])**-1 if x["sensit_units"] in ["m", "M"] else x["sensit_val"], axis=1)
simple_gain_vels = []
for _, row in db_gains_df.iterrows():
    sensit_val = row["sensit_val"]
    if row["sensit_units"] in ["m", "M"]:
        sensit_val = row["sensit_val"]*(2*np.pi*row["sensit_freq"])**-1
    elif row["sensit_units"] in ["m/s**2", "M/S**2"]:
        sensit_val = row["sensit_val"]*(2*np.pi*row["sensit_freq"])

    simple_gain_vels.append(sensit_val)
db_gains_df["simple_gain_vel"] = simple_gain_vels

# Save matching gain info

In [11]:
matching_gains = []
matching_gains_units = []
for i, row in db_gains_df.iterrows():
    chan_ondate = datetime.strptime(row["channel_ondate"], "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    chan_offdate = row["channel_offdate"]
    if type(chan_offdate) == str:    
        chan_offdate = datetime.strptime(chan_offdate, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    else:
        chan_offdate = None
    #print(chan_ondate, chan_offdate)
    matching_bb_df = gains_df[(gains_df["network"] == row["network"]) & 
                              (gains_df["station"] == row["station"]) & 
                              (gains_df["channel"] == row["seed_code"]) &
                              (chan_offdate is None or gains_df["mindate"] <= chan_offdate) &
                              (gains_df["maxdate"] >= chan_ondate)]
                                 # & (db_gains_df["sensit_units"].isin(["m", "M"]))]
                                 # & (db_gains_df["location"] == row["location"])]
    matching_gain = None
    matching_gain_units = None
    if len(matching_bb_df) > 0:
        uniq_vals = matching_bb_df["gain"].drop_duplicates().values
        print(row["network"], row["station"], row["seed_code"], row["location"], row["gain_vel"], chan_ondate, chan_offdate, len(matching_bb_df), uniq_vals)
        if len(uniq_vals) > 1:
            date_diffs = matching_bb_df["maxdate"] - chan_ondate
            matching_bb_df = matching_bb_df.iloc[np.argmax(date_diffs):np.argmax(date_diffs)+1]
            print("*", row["network"], row["station"], row["seed_code"], row["location"], row["gain_vel"], chan_ondate, chan_offdate, matching_bb_df["gain"].values)
        
        matching_gain = matching_bb_df["gain"].values[0]
        matching_gain_units = matching_bb_df["gain_units"].values[0]
        #print(matching_bb_df)

    matching_gains.append(matching_gain)
    matching_gains_units.append(matching_gain_units)

IW FLWY BH1 00 462428324.8058124 2016-10-22 04:00:00+00:00 2020-01-30 20:00:00+00:00 1 [4.62429e+08]
IW FLWY BH1 00 462428324.8058124 2020-01-30 20:00:00+00:00 2021-08-29 18:00:00+00:00 1 [4.62429e+08]
IW FLWY BH1 00 293919851.46844804 2021-08-29 18:00:00+00:00 None 1 [3.0172e+08]
IW FLWY BH2 00 462428324.8058124 2016-10-22 04:00:00+00:00 2020-01-30 20:00:00+00:00 1 [4.62429e+08]
IW FLWY BH2 00 462428324.8058124 2020-01-30 20:00:00+00:00 2021-08-29 18:00:00+00:00 1 [4.62429e+08]
IW FLWY BH2 00 293919851.46844804 2021-08-29 18:00:00+00:00 None 1 [3.0172e+08]
IW FLWY BHE    1145993158.470899 2007-03-16 16:46:00+00:00 2013-07-25 14:46:00+00:00 1 [1.25865e+09]
IW FLWY BHN    1145993158.470899 2007-03-16 16:46:00+00:00 2013-07-25 14:46:00+00:00 1 [1.25865e+09]
IW FLWY BHZ 00 1145993158.470899 2013-07-25 14:46:00+00:00 2016-10-22 04:00:00+00:00 1 [1.25865e+09]
IW FLWY BHZ 00 462428324.8058124 2016-10-22 04:00:00+00:00 2020-01-30 20:00:00+00:00 1 [4.62429e+08]
IW FLWY BHZ 00 462428324.8058124

In [12]:
db_gains_df["featmag_gain"] = matching_gains
db_gains_df["featmag_gain_units"] = matching_gains_units

In [13]:
for i, row in db_gains_df[~np.isnan(db_gains_df["featmag_gain"])].iterrows():
    gain_perc_err = (abs(row["gain_vel"] - row["featmag_gain"])/row["featmag_gain"])*100
    simple_gain_perc_err = (abs(row["simple_gain_vel"] - row["featmag_gain"])/row["featmag_gain"])*100
    print(f'{row["network"]}.{row["station"]}.{row["seed_code"]}.{row["location"]:5s}{ row["channel_ondate"]}-{row["channel_offdate"]} {gain_perc_err:.4f}% {simple_gain_perc_err:.4f}%')

IW.FLWY.BH1.00   2016-10-22 04:00:00-2020-01-30 20:00:00 0.0001% 0.0000%
IW.FLWY.BH1.00   2020-01-30 20:00:00-2021-08-29 18:00:00 0.0001% 0.0000%
IW.FLWY.BH1.00   2021-08-29 18:00:00-nan 2.5852% 2.5852%
IW.FLWY.BH2.00   2016-10-22 04:00:00-2020-01-30 20:00:00 0.0001% 0.0000%
IW.FLWY.BH2.00   2020-01-30 20:00:00-2021-08-29 18:00:00 0.0001% 0.0000%
IW.FLWY.BH2.00   2021-08-29 18:00:00-nan 2.5852% 2.5852%
IW.FLWY.BHE.     2007-03-16 16:46:00-2013-07-25 14:46:00 8.9506% 8.9509%
IW.FLWY.BHN.     2007-03-16 16:46:00-2013-07-25 14:46:00 8.9506% 8.9509%
IW.FLWY.BHZ.00   2013-07-25 14:46:00-2016-10-22 04:00:00 8.9506% 8.9509%
IW.FLWY.BHZ.00   2016-10-22 04:00:00-2020-01-30 20:00:00 0.0001% 0.0000%
IW.FLWY.BHZ.00   2020-01-30 20:00:00-2021-08-29 18:00:00 0.0001% 0.0000%
IW.FLWY.BHZ.00   2021-08-29 18:00:00-nan 2.5852% 2.5852%
IW.FLWY.BHZ.     2007-03-16 16:46:00-2013-07-25 14:46:00 8.9506% 8.9509%
IW.IMW.BH1.00   2013-07-25 15:11:00-2023-09-15 17:00:00 8.9506% 8.9509%
IW.IMW.BH2.00   2013-07-25 

In [14]:
net = "IW"
stat = "FLWY"
loc = "00"
chan = "BH2"
gains_df[(gains_df["network"] == net) & 
            (gains_df["station"] == stat) & 
            #(gains_df["location"] == loc) & 
            (gains_df["channel"] == chan)].sort_values("mindate")

,network,station,location,channel,gain,gain_units,channel_dip,channel_azimuth,mindate,maxdate
12322,IW,FLWY,00,BH2,462429000.0,DU/M/S,0.0,90.0,2016-11-17 06:08:44.755000+00:00,2021-07-31 08:11:14.434572+00:00
47608,IW,FLWY,00,BH2,301720000.0,DU/M/S,0.0,92.8,2021-09-08 17:01:40.424286+00:00,2023-12-17 03:22:16.120020+00:00


In [15]:
db_gains_df[(db_gains_df["network"] == net) & 
            (db_gains_df["station"] == stat) & 
            # (db_gains_df["location"] == loc) & 
            (db_gains_df["seed_code"] == chan)].sort_values("channel_ondate")

,channel_id,network,station,location,seed_code,channel_samp_rate,sensit_units,sensit_freq,sensit_val,gain_vel,receiver_lat,receiver_lon,channel_azimuth,channel_ondate,channel_offdate,simple_gain_vel,featmag_gain,featmag_gain_units
7,959,IW,FLWY,00,BH2,40.0,m/s,0.05,1.145990e+09,1.145993e+09,44.083002,-110.699888,90.0,2013-07-25 14:46:00,2016-10-22 04:00:00,1.145990e+09,NaN,None
8,569,IW,FLWY,00,BH2,40.0,m/s,0.02,4.624290e+08,4.624283e+08,44.083002,-110.699888,90.0,2016-10-22 04:00:00,2020-01-30 20:00:00,4.624290e+08,462429000.0,DU/M/S
9,378,IW,FLWY,00,BH2,40.0,m/s,0.02,4.624290e+08,4.624283e+08,44.083002,-110.699888,90.0,2020-01-30 20:00:00,2021-08-29 18:00:00,4.624290e+08,462429000.0,DU/M/S
10,5,IW,FLWY,00,BH2,40.0,m/s,0.02,2.939200e+08,2.939199e+08,44.083002,-110.699888,92.8,2021-08-29 18:00:00,NaN,2.939200e+08,301720000.0,DU/M/S


In [16]:
db_gains_df["sensit_units"].value_counts()

sensit_units
m         227
m/s       164
M/S        48
V          30
m/s**2      3
Name: count, dtype: int64

In [17]:
db_gains_df.to_csv("../files/db_featmag_gains_S.csv", index=False)